# Stage 1 — Regex Anonymization Pass

Reads from `data/preprocessed/<TICKER>/`, applies deterministic string substitutions
defined in `config/entities.yaml`, and writes to `data/intermediate/<TICKER>/`.

**Pipeline position:** `preprocessed` → **`intermediate`** → `processed`  
**Next step:** hand `data/intermediate/` to the agent layer (see `agent_layer.md`).


In [ ]:
#@title Setup { display-mode: "form" }
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"])
print("Dependencies ready.")


In [ ]:
#@title Imports & Paths
import re
import yaml
import collections
from pathlib import Path
from tqdm.notebook import tqdm

# All paths relative to this notebook (filing_anonymization/)
NOTEBOOK_DIR  = Path(".")
PREPROCESSED  = NOTEBOOK_DIR / "data" / "preprocessed"
INTERMEDIATE  = NOTEBOOK_DIR / "data" / "intermediate"
CONFIG_PATH   = NOTEBOOK_DIR / "config" / "entities.yaml"

with open(CONFIG_PATH) as f:
    CONFIG = yaml.safe_load(f)

print(f"Config loaded — {len(CONFIG['tickers'])} tickers:\n")
for ticker in CONFIG["tickers"]:
    n = len(list((PREPROCESSED / ticker).glob("*.txt")))
    syn = CONFIG["tickers"][ticker]["company_names"]["synthetic"]
    print(f"  {ticker:6s}  {n:4d} files   →   {syn}")


## Build Substitution Patterns

For each ticker we compile a sorted list of `(regex_pattern, replacement)` pairs.
Pairs are sorted **longest-real-string first** so that compound names (e.g. `Advanced Micro Devices, Inc.`)
are replaced before shorter sub-strings (`AMD`) — preventing partial-match collisions.

Single-token strings (no spaces or punctuation) get `\b` word-boundary anchors.
Multi-token strings (with spaces/commas) use literal escaped matching.


In [ ]:
#@title Substitution Builder

def _to_list(val):
    """Normalize yaml value → list, dropping None entries."""
    if val is None:
        return []
    if isinstance(val, list):
        return [str(v) for v in val if v is not None]
    return [str(val)]


def _add_pairs(pairs, real_val, synthetic, word_boundary=None):
    """Append (priority, pattern, replacement) tuples."""
    if not synthetic:
        return
    for real in _to_list(real_val):
        if not real:
            continue
        # Default: use word boundary only for single-token strings
        needs_wb = word_boundary
        if needs_wb is None:
            needs_wb = not any(c in real for c in ' ,()')
        if needs_wb:
            pattern = r'\b' + re.escape(real) + r'\b'
        else:
            pattern = re.escape(real)
        pairs.append((len(real), pattern, synthetic))


def build_substitutions(ticker_cfg):
    """Return sorted [(pattern, replacement)] list for one ticker."""
    pairs = []

    # Company names
    cn = ticker_cfg.get("company_names") or {}
    _add_pairs(pairs, cn.get("real"), cn.get("synthetic"))

    # Ticker symbol (always word-boundary)
    ts = ticker_cfg.get("ticker_symbol") or {}
    _add_pairs(pairs, ts.get("real"), ts.get("synthetic"), word_boundary=True)
    for ar in (ts.get("also_replace") or []):
        _add_pairs(pairs, ar.get("real"), ar.get("synthetic"), word_boundary=True)

    # Legal IDs (EIN / commission file / CIK)
    legal = ticker_cfg.get("legal_ids") or {}
    for key in ["ein", "commission_file", "cik"]:
        item = legal.get(key) or {}
        _add_pairs(pairs, item.get("real"), item.get("synthetic"))

    # Addresses and phones (no word-boundary forcing)
    for addr in (ticker_cfg.get("addresses") or []):
        _add_pairs(pairs, addr.get("real"), addr.get("synthetic"), word_boundary=False)
    for phone in (ticker_cfg.get("phone_numbers") or []):
        _add_pairs(pairs, phone.get("real"), phone.get("synthetic"), word_boundary=False)

    # Domains
    for domain in (ticker_cfg.get("domains") or []):
        _add_pairs(pairs, domain.get("real"), domain.get("synthetic"))

    # Executives
    for exec_e in (ticker_cfg.get("executives") or []):
        _add_pairs(pairs, exec_e.get("real"), exec_e.get("synthetic"))

    # All product/brand/banner/label categories
    for cat in ["products", "brands", "store_banners", "private_labels"]:
        for item in (ticker_cfg.get(cat) or []):
            _add_pairs(pairs, item.get("real"), item.get("synthetic"))

    # Subsidiaries
    for sub in (ticker_cfg.get("subsidiaries") or []):
        _add_pairs(pairs, sub.get("real"), sub.get("synthetic"))

    # Sort longest-first to prevent partial-match collisions
    pairs.sort(key=lambda x: x[0], reverse=True)
    return [(pat, repl) for _, pat, repl in pairs]


# Sanity check: print pattern count per ticker
print("Pattern counts per ticker:\n")
for ticker, cfg in CONFIG["tickers"].items():
    subs = build_substitutions(cfg)
    print(f"  {ticker:6s}  {len(subs):3d} patterns")


In [ ]:
#@title File Processing Functions

def apply_substitutions(text, substitutions):
    for pattern, replacement in substitutions:
        text = re.sub(pattern, replacement, text)
    return text


def process_file(in_path, out_path, substitutions):
    with open(in_path, "r", encoding="utf-8", errors="replace") as f:
        text = f.read()
    result = apply_substitutions(text, substitutions)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(result)
    return result


print("Processing functions ready.")


## Dry Run Preview

Before touching any files, scan the first 2 000 characters of a sample document
per ticker and show what the regex will replace.  
If a critical name isn't being caught, fix `config/entities.yaml` before running the main pass.


In [ ]:
#@title Dry Run (sample first file per ticker)
PREVIEW_CHARS = 3000  #@param {type:"integer"}

for ticker, cfg in CONFIG["tickers"].items():
    subs = build_substitutions(cfg)
    files = sorted((PREPROCESSED / ticker).glob("*.txt"))
    if not files:
        print(f"\n{ticker}: no files found")
        continue

    sample = files[0]
    with open(sample, encoding="utf-8", errors="replace") as f:
        snippet = f.read()[:PREVIEW_CHARS]

    hits = []
    for pattern, replacement in subs:
        for m in re.finditer(pattern, snippet):
            hits.append((m.start(), m.group(), replacement))
    hits.sort(key=lambda x: x[0])

    syn_name = cfg["company_names"]["synthetic"]
    print(f"\n{'='*64}")
    print(f"  {ticker}  →  {syn_name}   ({len(subs)} patterns)")
    print(f"  Sample: {sample.name}")
    if hits:
        print(f"  Matches in first {PREVIEW_CHARS} chars:")
        for pos, real, synth in hits[:10]:
            print(f"    pos {pos:5d}  '{real}'  →  '{synth}'")
        if len(hits) > 10:
            print(f"    ... and {len(hits)-10} more")
    else:
        print(f"  ⚠️  No matches in first {PREVIEW_CHARS} chars — check patterns")


## Main Pass: preprocessed → intermediate

Processes every `.txt` file across all 8 tickers.
Output mirrors the same directory structure under `data/intermediate/`.


In [ ]:
#@title Run All Tickers { display-mode: "form" }
TICKERS_TO_RUN = "ALL"  #@param {type:"string"}

tickers_to_run = (
    list(CONFIG["tickers"].keys())
    if TICKERS_TO_RUN.strip().upper() == "ALL"
    else [t.strip().upper() for t in TICKERS_TO_RUN.split(",")]
)

run_summary = {}

for ticker in tickers_to_run:
    if ticker not in CONFIG["tickers"]:
        print(f"⚠️  Unknown ticker '{ticker}' — skipping")
        continue

    cfg  = CONFIG["tickers"][ticker]
    subs = build_substitutions(cfg)
    in_dir  = PREPROCESSED  / ticker
    out_dir = INTERMEDIATE / ticker
    out_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(in_dir.glob("*.txt"))
    print(f"\n{ticker}: {len(files)} files  ({len(subs)} patterns)")

    for fpath in tqdm(files, desc=ticker, unit="file"):
        out_path = out_dir / fpath.name
        process_file(fpath, out_path, subs)

    run_summary[ticker] = len(files)
    print(f"  ✓ {ticker} complete")

print("\n" + "="*50)
print("Stage 1 complete — Summary:")
for ticker, n in run_summary.items():
    print(f"  {ticker:6s}  {n:4d} files  →  data/intermediate/{ticker}/")


## Validation

Scan every output file for leftover real company names (the full legal name and
its longest variant only — short aliases like `GM` are too noisy to scan for globally).

Files flagged here are candidates for the agent pass (Stage 2).
A clean run may still have residuals from contextual or paraphrased references —
that's expected and is what Stage 2 is for.


In [ ]:
#@title Validation Scan

# Use only the longest real-name variant per ticker (skip short tokens)
ANCHOR_NAMES = {}
for ticker, cfg in CONFIG["tickers"].items():
    reals = _to_list(cfg.get("company_names", {}).get("real"))
    candidates = [r for r in reals if len(r) > 6]
    if candidates:
        ANCHOR_NAMES[ticker] = sorted(candidates, key=len, reverse=True)[:3]

print("Scanning data/intermediate/ for leftover real names...\n")
residuals = collections.defaultdict(list)

for ticker, anchor_list in ANCHOR_NAMES.items():
    out_dir = INTERMEDIATE / ticker
    files   = sorted(out_dir.glob("*.txt"))
    if not files:
        continue

    for fpath in tqdm(files, desc=f"Validating {ticker}", unit="file"):
        with open(fpath, encoding="utf-8", errors="replace") as f:
            text = f.read()
        for anchor in anchor_list:
            if re.search(re.escape(anchor), text, re.IGNORECASE):
                residuals[ticker].append((fpath.name, anchor))
                break

print("\nValidation Results:")
if not any(residuals.values()):
    print("  ✓ No anchor names found in intermediate files.")
    print("  Intermediate data looks clean for Stage 2.")
else:
    total_files = sum(len(v) for v in residuals.values())
    print(f"  ⚠️  {total_files} file(s) still contain real identifiers (agent pass will handle):")
    for ticker, hits in residuals.items():
        print(f"\n  {ticker}: {len(hits)} file(s)")
        for fname, matched in hits[:5]:
            print(f"    {fname}  [matched: '{matched}']")
        if len(hits) > 5:
            print(f"    ... and {len(hits)-5} more")
    print("\n  → See agent_layer.md for Stage 2 instructions.")
